### from Stuart

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
"""
This cell does the initial project setup.
If you start a new script or notebook, make sure to copy & paste this part.

A script with this code uses the location of the `.env` file as the anchor for
the whole project (= PROJECT_ROOT). Afterwards, code inside the `src` directory
are available for import.
"""
from pathlib import Path
import sys
from dotenv import load_dotenv, find_dotenv
load_dotenv()
PROJECT_ROOT = Path(find_dotenv()).parent
sys.path.append(str(PROJECT_ROOT.joinpath('src')))
print(f"Project root directory: {PROJECT_ROOT}")

In [ ]:
from utils.config import CACHE_DIR, DATA_DIR, FIG_DIR
result_dir = FIG_DIR / 'quan_propagation'
cache_dir = CACHE_DIR / 'quan_propagation'

In [ ]:
import requests
import numpy as np
import pandas as pd
import pyarrow.feather as feather

from ngsidekick import segment_properties_to_dataframe, write_precomputed_annotations

In [ ]:
# It is convenient to make sure our ROI enumeration matches
# the segment IDs in the ROI layer, for easier copying and pasting.
brain_roi_info_url = 'https://storage.googleapis.com/flyem-male-cns/rois/fullbrain-roi-v4/segment_properties/info'
brain_roi_info = requests.get(brain_roi_info_url).json()
rois = segment_properties_to_dataframe(brain_roi_info)['source'].rename('roi')
rois = rois.reindex(np.arange(rois.index.max() + 1))
rois = rois.fillna(pd.Series([f'null_{i}' for i in rois.index]))
rois[0] = 'NotPrimary'
rois

In [ ]:
# df = feather.read_feather('retinotopy_tbar_olr.feather')
df = pd.read_feather(cache_dir / 'retinotopy_tbar_olr.feather')
df['roi'] = df['roi'].astype(pd.CategoricalDtype(rois))

# Make types compatible with neuroglancer properties.
df = df.astype({col: np.float32 for col in ['hex1', 'hex2', 'r', 'theta', 'v', 'h', 'r2']})

# We'll encode the body IDs twice: Once as a "relationship" (uint64)
# which allows annotations to be selected automatically via the neuroglancer UI,
# but again as a "property" (uint32) in case we want to somehow access the body IDs
# programmatically in the annotation shader.
df = df.rename(columns={'bodyId': 'body_pre'})
df['body'] = df['body_pre'].astype(np.uint32)

# We also encode the ROIs as a relationship.
df['roi_pre'] = df['roi'].cat.codes.astype(np.int8)

df.head()

In [ ]:
write_precomputed_annotations(
    df,
    {'names': [*'xyz'], 'scales': [8,8,8]},
    'point',
    ['hex1', 'hex2', 'body', 'roi', 'r2', 'r', 'theta', 'v', 'h'],
    ['body_pre', 'roi_pre'],
    output_dir='retinotopy_tbar_olr_points'
)

In [ ]:
# Upload to the bucket.
!gsutil -q -m cp -r retinotopy_tbar_olr_points gs://flyem-cns-roi-7c971aa681da83f9a074a1f0e8ef60f4/

In [ ]:
# Define variable names we can refer to in the shader.
roi_constants = [
    f"{roi}_{side}"
    if side else roi
    for roi, _, side in
        pd.Series(df['roi'].cat.categories)
        .str.extract(r'([^()]+)(\(([RL])\))?')
        .replace([np.nan], [None]).values
]

roi_constants = [s.replace('-', '_').replace("'", 'p') for s in roi_constants]
print(roi_constants)

In [ ]:
for i, constant in enumerate(roi_constants):
    if 'null' in constant or 'NotPrimary' in constant:
        print(f"bool roi_{constant} = false;")
    else:
        print(f"#uicontrol bool roi_{constant} checkbox(default=true)")

print("\n")
constant_list = ', '.join([f'roi_{constant}' for constant in roi_constants])
print("bool roi_selections[] = bool[](" + constant_list + ");")


# Scratch - color conversion functions

In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt
from colormath.color_objects import LabColor, XYZColor, sRGBColor
from colormath.color_conversions import convert_color

In [ ]:
def disk_to_lab(r, theta, L=75):
    """
    Map a point in the unit disk (r, theta) to the CIE LAB color space with fixed L* and a 45° rotation.
    
    Parameters:
    r: float
        Radius in the unit disk (0 to 1)
    theta: float
        Angle in radians (0 to 2π)
    L: float
        Fixed L* value (lightness) in LAB color space (0 to 100)
    
    Returns:
    tuple: (L, a, b) coordinates in the CIE LAB color space
    """
    # Input validation
    if not (0 <= r <= 1):
        raise ValueError("r must be in the range [0, 1]")
    # Scale the radius to get an appropriate range for a* and b*
    # At typical L=75, the a* and b* ranges are roughly [-100, 100]
    max_radius = 100  # Maximum value for a* and b*
    
    # Convert polar coordinates to a* and b* coordinates
    a = r * max_radius * np.cos(theta + np.pi/4)
    b = r * max_radius * np.sin(theta + np.pi/4)
    
    return L, a, b

def lab_to_rgb(L, a, b):
    """
    Convert LAB color to sRGB for display.
    
    Parameters:
    L, a, b (float): CIE LAB color coordinates
    
    Returns:
    tuple: (R, G, B) values in range [0, 1], or None if the color is out of gamut
    """
    try:
        # Create a LAB color object
        lab = LabColor(L, a, b, illuminant="d50", observer="2")
        
        # Convert to XYZ
        xyz = convert_color(lab, XYZColor)
        
        # Convert to sRGB
        rgb = convert_color(xyz, sRGBColor)
        
        # Get RGB values
        rgb_values = (rgb.rgb_r, rgb.rgb_g, rgb.rgb_b)
        
        # Check if the color is in the sRGB gamut
        if (0 <= rgb.rgb_r <= 1) and (0 <= rgb.rgb_g <= 1) and (0 <= rgb.rgb_b <= 1):
            return rgb_values
        else:
            # Color is out of gamut
            print(f"LAB({L:.1f}, {a:.1f}, {b:.1f}) is out of sRGB gamut. Clamping to [0, 1].")
            return (rgb.clamped_rgb_r, rgb.clamped_rgb_g, rgb.clamped_rgb_b)
    except:
        # Exception during conversion (likely out of gamut)
        print(f"Exception converting LAB({L:.1f}, {a:.1f}, {b:.1f}) to sRGB.")
        return None

In [ ]:

# Use RdYlBu divergent colormap (Red-Yellow-Blue)
# Other options: 'RdBu', 'RdYlGn', 'PiYG', 'PRGn', 'BrBG', 'PuOr', 'Spectral'
cmap = plt.cm.RdYlBu  # Updated syntax for newer matplotlib versions

# normalize to 0-1 range and get hex color

h_norm = (df['h'] - df['h'].min()) / (df['h'].max() - df['h'].min())
df['color_h'] = [mpl.colors.to_hex(color) for color in cmap(h_norm)]

v_norm = (df['v'] - df['v'].min()) / (df['v'].max() - df['v'].min())
df['color_v'] = [mpl.colors.to_hex(color) for color in cmap(v_norm)]

In [ ]:
df['h'].min(), df['h'].max()

In [ ]:
df['v'].min(), df['v'].max()

In [ ]:
df['r2'].min(), df['r2'].max()

In [ ]:
float h_min = -15.50169;
float_h_max = 17.038269;

float v_min = -27.799099;
float v_max = 32.45468;